In [0]:
# D3 - Customer Profile & Behavior
# Sources: sales_project_db.silver.fact_sales, sales_project_db.silver.dim_customers
# PySpark equivalent of the original SQL query (D3 - Customer Profile & Behavior.dbquery)
 
from pyspark.sql import functions as f
 
fact_sales = spark.table("sales_project_db.silver.fact_sales")
dim_customers = spark.table("sales_project_db.silver.dim_customers")
 
# Step 1: per-customer spend and age (mirrors the inner subquery in the SQL)
customer_spend = (
    fact_sales.alias("f")
    .join(
        dim_customers.alias("c"),
        f.col("f.customer_key") == f.col("c.customer_key"),
        "inner",
    )
    .filter(f.col("c.birthdate").isNotNull())
    .withColumn(
        "customer_age",
        f.year(f.current_date()) - f.year(f.col("c.birthdate")),
    )
    .groupBy(f.col("c.customer_number").alias("customer_number"), "customer_age")
    .agg(f.sum("f.sales_amount").alias("total_customer_spend"))
)
 
# Step 2: bucket into age groups and aggregate (mirrors the outer query)
df_customer_profile = (
    customer_spend
    .withColumn(
        "age_group",
        f.when(f.col("customer_age") < 30, "Under 30")
        .when((f.col("customer_age") >= 30) & (f.col("customer_age") <= 39), "30-39")
        .when((f.col("customer_age") >= 40) & (f.col("customer_age") <= 49), "40-49")
        .when((f.col("customer_age") >= 50) & (f.col("customer_age") <= 59), "50-59")
        .otherwise("60 and over"),
    )
    .groupBy("age_group")
    .agg(
        f.countDistinct("customer_number").alias("total_customers"),
        f.round(f.sum("total_customer_spend"), 2).alias("total_revenue"),
        f.round(f.avg("total_customer_spend"), 2).alias("average_spend_per_customer"),
    )
    .orderBy(f.col("age_group").asc())
)
 
display(df_customer_profile)

Databricks visualization. Run in Databricks to view.